# Aggregation Quality Analysis

This notebook demonstrates how to analyze the quality of time series aggregation using tsam's built-in plotting tools.

We will:
1. Load and aggregate time series data
2. Visualize the original vs reconstructed data
3. Analyze cluster structure and assignments
4. Examine residuals and error patterns
5. Compare different aggregation configurations

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, ExtremeConfig

pio.renderers.default = "notebook"

## 1. Load Data and Run Aggregation

In [ ]:
# Load test data (8760 hours = 1 year of hourly data)
raw = pd.read_csv("testdata.csv", index_col=0)
print(f"Data shape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
raw.head()

In [ ]:
# Run aggregation with 12 typical days
result = tsam.aggregate(
    raw,
    n_clusters=12,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
)

print(f"Number of clusters: {result.n_clusters}")
print(f"Timesteps per period: {result.n_timesteps_per_period}")
print(f"Total original periods: {len(raw) // result.n_timesteps_per_period}")

## 2. Visual Comparison: Original vs Reconstructed

### Heatmaps

Heatmaps show the full year with periods (days) on the x-axis and timesteps (hours) on the y-axis.

In [ ]:
# Original data heatmap
result.plot.heatmap(column="T", use_original=True, title="Original Temperature")

In [ ]:
# Reconstructed data heatmap (same color scale for comparison)
result.plot.heatmap(column="T", title="Reconstructed Temperature")

In [ ]:
# Multi-column heatmaps for an overview
result.plot.heatmaps(columns=["GHI", "T", "Load"], title="Reconstructed Time Series")

### Duration Curves

Duration curves show sorted values and reveal how well the aggregation preserves the value distribution.

In [ ]:
# Compare original vs reconstructed duration curves
result.plot.compare(columns=["Load"], mode="duration_curve")

In [ ]:
# Duration curves for all columns
result.plot.duration_curve()

### Time Slice Comparison

Zoom into specific time periods to see how well the aggregation captures temporal patterns.

In [ ]:
# Compare a week in winter
result.plot.compare(
    columns=["T", "Load"],
    mode="overlay",
    start="20100115",
    end="20100122",
    title="Winter Week Comparison",
)

In [ ]:
# Side-by-side view
result.plot.compare(
    columns=["GHI"],
    mode="side_by_side",
    start="20100701",
    end="20100708",
    title="Summer Week - Solar Irradiance",
)

## 3. Cluster Analysis

Understanding the cluster structure helps assess whether the aggregation captures meaningful patterns.

In [ ]:
# Cluster weights - how many days are represented by each typical day
result.plot.cluster_weights()

In [ ]:
# Cluster assignments - which cluster each original day belongs to
result.plot.cluster_assignments()

In [ ]:
# Representative profiles for each cluster
result.plot.cluster_representatives(columns=["T"])

In [ ]:
# Representative profiles for solar irradiance
result.plot.cluster_representatives(columns=["GHI"])

## 4. Error Analysis

### Accuracy Metrics

In [ ]:
# Overall accuracy metrics
print("Accuracy Summary:")
print(result.accuracy)
print("\nRMSE per column:")
print(result.accuracy.rmse)
print("\nMAE per column:")
print(result.accuracy.mae)

In [ ]:
# Visual comparison of accuracy metrics
result.plot.accuracy()

### Residual Analysis

Residuals (original - reconstructed) reveal where the aggregation performs well or poorly.

In [ ]:
# Residuals over time - look for systematic patterns
result.plot.residuals(columns=["Load"], mode="time_series")

In [ ]:
# Residual distribution - should be centered around zero
result.plot.residuals(columns=["T", "Load"], mode="histogram")

In [ ]:
# Error by period - which days have the highest reconstruction error
result.plot.residuals(columns=["Load"], mode="by_period")

In [ ]:
# Error by timestep - which hours within the day have highest error
result.plot.residuals(columns=["Load", "GHI"], mode="by_timestep")

## 5. Comparing Aggregation Configurations

Let's compare different numbers of clusters to see the accuracy-complexity tradeoff.

In [ ]:
# Run aggregations with different cluster counts
results = {}
for n in [4, 8, 12, 24]:
    results[f"{n} clusters"] = tsam.aggregate(
        raw,
        n_clusters=n,
        period_duration=24,
        cluster=ClusterConfig(method="hierarchical"),
    )

# Print accuracy comparison
print("RMSE comparison (Load):")
for name, res in results.items():
    print(f"  {name}: {res.accuracy.rmse['Load']:.2f}")

In [ ]:
# Compare duration curves across configurations
comparison_data = {"Original": raw}
for name, res in results.items():
    comparison_data[name] = res.reconstruct()

tsam.plot.compare(
    comparison_data,
    column="Load",
    plot_type="duration_curve",
    title="Duration Curve: Cluster Count Comparison",
)

In [ ]:
# Compare a specific time slice
tsam.plot.compare(
    comparison_data,
    column="Load",
    plot_type="time_slice",
    start="20100601",
    end="20100608",
    title="June Week: Cluster Count Comparison",
)

## 6. Effect of Extreme Period Preservation

Compare aggregation with and without preserving extreme values.

In [ ]:
# Without extreme preservation
result_no_extremes = tsam.aggregate(
    raw,
    n_clusters=8,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
)

# With extreme preservation
result_with_extremes = tsam.aggregate(
    raw,
    n_clusters=8,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
    extremes=ExtremeConfig(
        method="new_cluster",
        min_value=["T"],
        max_value=["Load", "GHI"],
    ),
)

print("Without extremes - Load RMSE:", result_no_extremes.accuracy.rmse["Load"])
print("With extremes - Load RMSE:", result_with_extremes.accuracy.rmse["Load"])

In [ ]:
# Compare peak preservation in duration curves
tsam.plot.compare(
    {
        "Original": raw,
        "No extremes": result_no_extremes.reconstruct(),
        "With extremes": result_with_extremes.reconstruct(),
    },
    column="Load",
    plot_type="duration_curve",
    title="Effect of Extreme Period Preservation on Load",
)

In [ ]:
# Compare temperature extremes
tsam.plot.compare(
    {
        "Original": raw,
        "No extremes": result_no_extremes.reconstruct(),
        "With extremes": result_with_extremes.reconstruct(),
    },
    column="T",
    plot_type="duration_curve",
    title="Effect of Extreme Period Preservation on Temperature",
)

## Summary

Key takeaways for evaluating aggregation quality:

1. **Heatmaps** show overall temporal patterns - look for missing features in reconstructed data
2. **Duration curves** reveal how well the value distribution is preserved - check peak/valley capture
3. **Cluster weights** show balance - very unequal weights may indicate suboptimal clustering
4. **Residual analysis** identifies systematic errors:
   - By period: which days are poorly represented
   - By timestep: which hours within the day have issues
5. **Extreme preservation** is critical for applications where peak values matter (e.g., capacity planning)